# Applications of Data Science - Summer Term 2026
- Group: 11 - International Taxation
- Group Members:
    - Simon Andreas **ERTL**
    - Berkay **KAYA**
    - Joel **PUTHENPARAMBIL**
    - Fedor **SAMOROKOV**
- Author of this notebook: Simon Andreas **ERTL**
## Group Project: Building an Austrian Tax Law Assistant
This project is split into four parts:
- Creation of a shared corpus regarding Austrian Tax Law from which each group can use for their LLM models. This corpus consists of questions regarding Austrian Tax Law focusing on different topics (e. g. International Taxation), detailed answers to these questions, and sources referenced.
- Applying models to the created corpus. This task itself is split into three parts:
    - Prompt LLMs (Inference): Out-of-the-box Large Language Models should be used to answer the questions created during Task 1.
    - Fine-tune models (Training): The models should then be fine-tuned using additional materials.
    - RAG: Using RAG, the performnance of the models should again be improved.
- Evalution of model performance
- Presentation

This notebook was created in Google Colab using the available T4 GPU architecture. To replicate the results, upload this notebook to Google Colab, and select the right architecture in the top right corner.

### Task 2
Before the tasks itself can be performed, various steps have to be completed beforehand:
- Importing necessary libaries
- Path handling
- Set up of the environment

In the following cell, the filename of the dataset (input and output) can be specified. Only the file name is needed, as the filepath will be handled seperately.

In [ ]:
input_dataset = "dataset_clean.csv" # Defining input dataset file name
output_dataset = "output_model1_ERTL.csv" # Defining output dataset file name

In the following code cells, the necessary libraries will be imported.

In [ ]:
# Taken from unsloth documentation
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers "trl<0.9.0" peft accelerate bitsandbytes

In [ ]:
# Importing libraries
import pandas as pd # For dataframe handling
from pathlib import Path # For path handling
import torch # For back-end
from unsloth import FastLanguageModel # To interact with the models
from google.colab import files # For downloading the files
from time import sleep # For pause before download

The cell below handles the filepath of the input and output files.

In [ ]:
PROJECT_CWD = Path.cwd() # Reading the current working directory
DATASET_DIR = PROJECT_CWD / input_dataset # Setting the path for the dataset
OUTPUT_DIR = PROJECT_CWD / output_dataset # Setting path for the output dataset

Now, the dataset will be read into a Pandas dataframe for easier handling.

In [ ]:
# Importing the dataset as a Pandas-dataframe
df = pd.read_csv(filepath_or_buffer = DATASET_DIR) # Reading the dataset file

### Task 2.1: Inference - Answering prompts from Task 1
In Task 1, each group had to formulate questions and answers (with sources) regarding the Austrian Tax Law, with each group having a different focus. Our grouped focused on international taxation.
In this task, Large Language Models should be implemented so they can answer the questions produced in Task 1. Task 2a requests basic inference: Out-of-the-box LLMs should answer the questions regarding Austrian Tax Law.

To configure the basic LLM model, we will first configure some parameters.

In [ ]:
MODEL_NAME = "unsloth/gemma-3-4b-it-unsloth-bnb-4bit" # Define model name
MAX_SEQ_LENGTH = 2048 # Setting the maximal context length
MAX_NEW_TOKENS = 512 # Setting the amount of new tokens generated
BATCH_SIZE = 8 # Setting batch size
DTYPE = None # Let computer choose FP accuracy
LOAD_IN_4_BIT = True # Loading in 4 bit to reduce memory usage
SEED = 30112003 # Seed for reproduceability

# Setting seed (depends on CUDA availability, which is available in Google Colab)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

Now we will load and configure the model and tokenizer for further use. We start by getting the `model` and `tokenizer` objects.

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained( # Load model and tokenizer
    model_name = MODEL_NAME, # Model which should be used
    max_seq_length = MAX_SEQ_LENGTH, # Maximal context length
    dtype = DTYPE, # Dtype preference; none in this case; determines precision
    load_in_4bit = LOAD_IN_4_BIT # Load model in 4 bit for better hardware performance
)

Now we will prepare the model for inference.

In [ ]:
FastLanguageModel.for_inference(model) # Switches into inference mode

In [ ]:
model.eval() # Switches into evaluation mode

Now we want to make sure that the model uses the correct architecture.

In [ ]:
DEVICE = next(model.parameters()).device # Checking which device torch runs on
DEVICE

Now we will configure a system prompt. With this, the LLM knows how to "behave".

In [ ]:
SYSTEM_PROMPT = """Du bist ein Assistent für österreichisches Steuerrecht.
Beantworte Fragen präzise, knapp und fachlich, wenn möglich in maximal 4-5 Sätzen, korrekt auf Deutsch.
Wenn die Rechtslage unklar ist oder dir Informationen fehlen, sage das offen.
Erfinde keine Gesetzesstellen oder Fakten. Zitiere verwendetete Paragraphen und Richtlinien des österreichen Rechts nach gültigen Zitierregeln.
"""

Now, we will define a `build_prompt_question`-function that will create the prompt for a specific question.

In [ ]:
def build_prompt_question(question: str): # Formats the prompts for use
    # As the model is multimodal, the type of the input has to be specified
    return [
        {"role": "system",
         "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {"role": "user",
         "content": [{"type": "text", "text": str(question).strip()}]}
    ]

Now we will define a function that will answer the questions in a batched manner.

In [ ]:
def answer_questions_batched(questions, batch_size: int = BATCH_SIZE): # Function to answer the questions
    questions = list(questions) # Make sure "questions" is a list
    conversations = [build_prompt_question(q) for q in questions] # Build prompt for every question
    answers = [None] * len(conversations) # Create None-list with length of no. of questions

    # Apply padding
    ## Some tokenizers have the tokenizer in a additional attribute
    if hasattr(tokenizer, "tokenizer"): # If in additional attribute
        tokenizer.tokenizer.padding_side = "left" # Apply left padding
        if tokenizer.tokenizer.pad_token is None: # If Padding Token is None
            tokenizer.tokenizer.pad_token = tokenizer.tokenizer.eos_token # Set Padding Token to EOS token
        eos_id = tokenizer.tokenizer.eos_token_id # Set EOS ID
    else:
        tokenizer.padding_side = "left" # Apply left padding
        if tokenizer.pad_token is None: # If no padding token is set
            tokenizer.pad_token = tokenizer.eos_token # Set Padding token to EOS token
        eos_id = tokenizer.eos_token_id # Set EOS ID

    # This batches the question answering
    for batch_start in range(0, len(conversations), batch_size): # Range from 0 to n with batch_size spacing
        batch_convs = conversations[batch_start: batch_start + batch_size] # Get conversations for batch

        # Apply chat template to batch
        inputs = tokenizer.apply_chat_template(
            batch_convs, # The conversations
            tokenize = True, # Tokenizes the conversations
            add_generation_prompt = True, # Adds marker for completion
            return_dict = True, # Request dictionairy output; for accessing input_ids and attention_mask
            return_tensors = "pt", # Return PyTorch Tensors
            padding = True, # Apply padding (left as set before)
            truncation = True, # Cut off when too long
        ).to(DEVICE) # Send to device

        input_len = inputs["input_ids"].shape[1] # Get size of a answer (all same length due to padding)

        # Actually answer the questions
        ## Inference mode
        with torch.inference_mode():
            outputs = model.generate(
                **inputs, # Apply inputs from tokenizer
                max_new_tokens = MAX_NEW_TOKENS, # Maximum no. of new tokens
                do_sample = False, # Greedy decoing, better for reproduceability
                use_cache = True, # Use caching for better performance
                pad_token_id = eos_id, # Set Padding Token
                eos_token_id = eos_id, # Set EOS token
            )

        generated_only = outputs[:, input_len:] # Only keep generated answers
        decoded = tokenizer.batch_decode( # Decode answers
            generated_only,
            skip_special_tokens = True,
        )

        for i, answer in enumerate(decoded):# Go through decoded answers
            answers[batch_start + i] = answer.strip() # Clean answer and add to answers list with right index

    return answers # Return all answers

Now that we have defined to logic, we can combine the functions to start the generation of the answers.

In [ ]:
df_prompts = df["prompt"].tolist() # Convert prompts-column to list

In [ ]:
prompt_answers = answer_questions_batched(df_prompts, batch_size = BATCH_SIZE) # Generate answers in batches

Now we can add the generated answers back to the dataframe.

In [ ]:
df["answer"] = prompt_answers # Add answers back to dataframe

The ID and answer columns can now be exported.

In [ ]:
# Output only the desired columns ID and answer
df[["id", "answer"]].to_csv(OUTPUT_DIR, # to set output directory
                            index = False, # No index wanted
                            quotechar = '"') # Wrap cell text in quotes

Now we wait for a few seconds until Colab has processed the file.

In [ ]:
sleep(10) # Wait for 10 seconds until Colab has processed the file

Now we can download the results.

In [ ]:
files.download(OUTPUT_DIR.resolve()) # Automatically download file